In [0]:
import pyspark.sql.functions as F

# 1. Carregando as tabelas da Silver
df_pedidos = spark.table("visagio_rocket_lab.silver.fat_pedidos")
df_reviews = spark.table("visagio_rocket_lab.silver.tb_order_reviews")
df_produtos = spark.table("visagio_rocket_lab.silver.dim_produtos")
df_vendedores = spark.table("visagio_rocket_lab.silver.dim_vendedores")

# 2. Join Consolidado 
df_analise = df_pedidos.join(df_reviews, df_pedidos.id_pedido == df_reviews.order_id, "inner") \
    .join(df_produtos, "id_produto", "inner") \
    .join(df_vendedores, df_pedidos["seller_id"] == df_vendedores["seller_id"], "inner")

# 3. Entrega 1 - Tabela Principal de Satisfação
df_gold_satisfacao = df_analise.groupBy("nome_categoria", "cidade") \
    .agg(
        F.count("review_id").alias("total_avaliacoes"),
        F.round(F.avg("review_score"), 2).alias("avaliacao_media"),
        F.sum(F.when(F.col("review_score") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
        F.sum(F.when(F.col("review_score") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
    ) \
    .withColumn("percentual_satisfacao", F.round((F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes")) * 100, 2))

# Salvando a tabela final na Gold
df_gold_satisfacao.write.mode("overwrite").saveAsTable("visagio_rocket_lab.gold.fat_avaliacoes_clientes")

# 4. Entrega 2 - Rankings de Qualidade (Exibição via display)
print("--- PRODUTO MAIS BEM AVALIADO ---")
display(df_analise.groupBy("id_produto").agg(F.avg("review_score").alias("media"), F.count("review_id").alias("vol")).orderBy(F.desc("media"), F.desc("vol")).limit(1))

print("--- VENDEDOR MAIS BEM AVALIADO ---")
# Especificando df_pedidos["seller_id"] para o agrupamento
display(df_analise.groupBy(df_pedidos["seller_id"]).agg(F.avg("review_score").alias("media"), F.count("review_id").alias("vol")).orderBy(F.desc("media"), F.desc("vol")).limit(1))

print("Gold finalizada com sucesso!")

--- PRODUTO MAIS BEM AVALIADO ---


id_produto,media,vol
37eb69aca8718e843d897aa7b82f462d,5.0,15


--- VENDEDOR MAIS BEM AVALIADO ---


seller_id,media,vol
48efc9d94a9834137efd9ea76b065a38,5.0,34


Gold finalizada com sucesso!
